### This RAG system provides a robust solution for querying oil and gas well data 
### Proper error handling and enhanced functionality. 
### The system automatically creates sample data if it doesn't exist 
### The program handles a wide variety of natural language queries about well locations, types, and status

#### Keith R. Holdaway FGS

### This complete RAG system:

#### Integrates with both CSV and SQLite data sources

#### Handles complex natural language queries

#### Uses spaCy for advanced NLP processing

#### Combines semantic search with structured database queries

#### Provides comprehensive responses with relevance scoring

In [1]:
# brownfield_rag_fixed.py
#!/usr/bin/env python3
"""
Comprehensive RAG System for Brownfield Well Management
With proper imports and error handling
"""

import pandas as pd
import sqlite3
import numpy as np
import re
import spacy
import json
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Any, Tuple, Optional
import warnings
import logging
import os
import sys

# Set up logging with console output
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("BrownfieldRAG")

# Suppress specific warnings
warnings.filterwarnings("ignore", message=".*The model you're using has no word vectors loaded.*")

def print_progress(message):
    """Print progress messages with formatting"""
    print(f"\n{'-'*60}")
    print(f"{message}")
    print(f"{'-'*60}")

class BrownfieldRAGSystem:
    """Comprehensive RAG system for brownfield well management"""
    
    def __init__(self, data_dir: str = "data"):
        print_progress("Initializing Brownfield RAG System")
        
        self.data_dir = data_dir
        self.setup_data_directories()
        
        # Initialize NLP components
        self.nlp = self.initialize_nlp()
        if self.nlp is None:
            print("ERROR: Could not initialize spaCy. Please install with: python -m spacy download en_core_web_sm")
            return
        
        # Initialize data connectors
        print_progress("Initializing data sources")
        self.well_data = self.initialize_data_sources()
        
        # Prepare document corpus for semantic search
        print_progress("Preparing document corpus")
        self.documents, self.doc_vectors, self.vectorizer = self.prepare_document_corpus()
        
        print_progress("Brownfield RAG System initialized successfully")
    
    def setup_data_directories(self):
        """Create necessary data directories if they don't exist"""
        os.makedirs(self.data_dir, exist_ok=True)
        os.makedirs(f"{self.data_dir}/csv", exist_ok=True)
        os.makedirs(f"{self.data_dir}/sqlite", exist_ok=True)
        os.makedirs(f"{self.data_dir}/json", exist_ok=True)
        print(f"Created data directories in {self.data_dir}")
    
    def initialize_nlp(self):
        """Initialize spaCy NLP model with fallback options"""
        try:
            nlp = spacy.load("en_core_web_sm")
            print("✓ Loaded spaCy model: en_core_web_sm")
            return nlp
        except OSError:
            try:
                nlp = spacy.load("en_core_web_md")
                print("✓ Loaded spaCy model: en_core_web_md")
                return nlp
            except OSError:
                print("✗ Could not load any spaCy model")
                return None
    
    def initialize_data_sources(self):
        """Initialize and populate all data sources"""
        data_sources = {}
        
        # Initialize CSV data
        data_sources['csv'] = CSVDataConnector(f"{self.data_dir}/csv/well_data.csv")
        data_sources['csv'].create_sample_data()
        
        # Initialize SQLite database
        data_sources['sqlite'] = SQLiteDataConnector(f"{self.data_dir}/sqlite/brownfield_wells.db")
        data_sources['sqlite'].create_sample_data()
        
        # Initialize JSON data
        data_sources['json'] = JSONDataConnector(f"{self.data_dir}/json/well_metadata.json")
        data_sources['json'].create_sample_data()
        
        return data_sources
    
    def prepare_document_corpus(self):
        """Prepare document corpus for semantic search from all data sources"""
        documents = []
        
        # Get data from all sources
        csv_data = self.well_data['csv'].load_data()
        sql_data = self.well_data['sqlite'].load_data()
        json_data = self.well_data['json'].load_data()
        
        print(f"Loaded {len(csv_data)} records from CSV, {len(sql_data)} from SQLite, {len(json_data)} from JSON")
        
        # Create document representations
        for _, row in csv_data.iterrows():
            doc = self.create_well_document(row, "CSV")
            documents.append(doc)
        
        for _, row in sql_data.iterrows():
            doc = self.create_well_document(row, "SQLite")
            documents.append(doc)
        
        # Handle JSON data
        for well_id, metadata in json_data.items():
            doc = self.create_metadata_document(well_id, metadata)
            documents.append(doc)
        
        print(f"Created {len(documents)} document representations")
        
        # Vectorize documents
        vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
        doc_vectors = vectorizer.fit_transform(documents)
        
        print("Document vectorization completed")
        return documents, doc_vectors, vectorizer
    
    def create_well_document(self, row: pd.Series, source: str) -> str:
        """Create a text document representation of a well"""
        doc_parts = [
            f"Well {row.get('name', row.get('well_name', 'Unknown'))}",
            f"located at coordinates (X: {row.get('x_coord', 'N/A')}, Y: {row.get('y_coord', 'N/A')})",
            f"in Sector {row.get('sector', 'N/A')}",
            f"Type: {row.get('well_type', 'N/A')}",
            f"Status: {row.get('status', 'N/A')}",
            f"Production rate: {row.get('production_rate', 'N/A')} BOPD",
            f"Drilled on: {row.get('drill_date', 'N/A')}",
            f"Reservoir: {row.get('reservoir', 'N/A')}",
            f"Data source: {source}"
        ]
        return ". ".join(doc_parts)
    
    def create_metadata_document(self, well_id: str, metadata: Dict) -> str:
        """Create a text document from JSON metadata"""
        doc_parts = [f"Metadata for well {well_id}"]
        
        if 'description' in metadata:
            doc_parts.append(f"Description: {metadata['description']}")
        if 'completion_data' in metadata:
            doc_parts.append(f"Completion data: {metadata['completion_data']}")
        if 'production_history' in metadata:
            doc_parts.append(f"Production history: {metadata['production_history']}")
        if 'geological_data' in metadata:
            doc_parts.append(f"Geological data: {metadata['geological_data']}")
        
        return ". ".join(doc_parts)
    
    def parse_query(self, query: str) -> Dict[str, Any]:
        """Parse natural language queries using spaCy"""
        if not self.nlp:
            return {"type": "general", "filters": {}}
        
        doc = self.nlp(query.lower())
        query_type = "general"
        filters = {}
        
        # Check for location queries
        location_terms = ['where', 'location', 'coordinate', 'position', 'sector']
        if any(term in query.lower() for term in location_terms):
            query_type = "location"
        
        # Check for status queries
        status_terms = ['status', 'active', 'inactive', 'drilling']
        if any(term in query.lower() for term in status_terms):
            query_type = "status"
        
        # Check for production queries
        production_terms = ['production', 'rate', 'barrel', 'bopd']
        if any(term in query.lower() for term in production_terms):
            query_type = "production"
        
        # Check for type queries
        type_terms = ['type', 'injection', 'monitoring', 'production well']
        if any(term in query.lower() for term in type_terms):
            query_type = "well_type"
        
        # Check for date queries
        date_terms = ['date', 'drilled', 'completed', 'year', 'month', 'recent', 'new']
        if any(term in query.lower() for term in date_terms):
            query_type = "date"
        
        # Check for reservoir queries
        reservoir_terms = ['reservoir', 'formation', 'sandstone', 'limestone', 'shale']
        if any(term in query.lower() for term in reservoir_terms):
            query_type = "reservoir"
        
        # Extract entities
        for ent in doc.ents:
            if ent.label_ == "GPE" or ent.label_ == "LOC":
                filters['location'] = ent.text
            elif ent.label_ == "DATE":
                filters['date'] = ent.text
        
        # Extract well types
        if "production" in query.lower():
            filters['well_type'] = 'production'
        elif "injection" in query.lower():
            filters['well_type'] = 'injection'
        elif "monitoring" in query.lower():
            filters['well_type'] = 'monitoring'
        
        # Extract reservoir types
        if "sandstone" in query.lower():
            filters['reservoir'] = 'Sandstone'
        elif "limestone" in query.lower():
            filters['reservoir'] = 'Limestone'
        elif "shale" in query.lower():
            filters['reservoir'] = 'Shale'
        
        # Extract sectors
        sector_match = re.search(r'[a-z]-?\d+', query, re.IGNORECASE)
        if sector_match:
            filters['sector'] = sector_match.group(0).upper()
        
        return {"type": query_type, "filters": filters}
    
    def process_query(self, query: str) -> Dict[str, Any]:
        """Process a natural language query and return a comprehensive response"""
        print(f"\nProcessing query: '{query}'")
        
        # Parse the query
        parsed_query = self.parse_query(query)
        print(f"Parsed as: {parsed_query['type']} query with filters: {parsed_query['filters']}")
        
        # Retrieve relevant information
        semantic_results = self.semantic_search(query)
        structured_results = self.structured_query(parsed_query)
        
        # Generate response
        response = self.generate_response(query, parsed_query, semantic_results, structured_results)
        
        return response
    
    def semantic_search(self, query: str, top_k: int = 5) -> List[Tuple[str, float]]:
        """Perform semantic search on document corpus"""
        query_vector = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vector, self.doc_vectors)[0]
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        return [(self.documents[i], similarities[i]) for i in top_indices]
    
    def structured_query(self, parsed_query: Dict) -> Dict[str, Any]:
        """Execute structured queries against all data sources"""
        results = {}
        
        # Query each data source
        for source_name, connector in self.well_data.items():
            try:
                if hasattr(connector, 'query'):
                    result = connector.query(parsed_query['filters'])
                    # Handle different return types
                    if isinstance(result, pd.DataFrame):
                        results[source_name] = result
                    elif isinstance(result, dict):
                        # Convert dict to DataFrame for consistency
                        if result:
                            results[source_name] = pd.DataFrame.from_dict(result, orient='index').reset_index()
                        else:
                            results[source_name] = pd.DataFrame()
                    else:
                        results[source_name] = pd.DataFrame()
                else:
                    # Fallback to loading all data and filtering
                    data = connector.load_data()
                    if isinstance(data, pd.DataFrame):
                        results[source_name] = self.apply_filters(data, parsed_query['filters'])
                    else:
                        results[source_name] = pd.DataFrame()
            except Exception as e:
                print(f"Error querying {source_name}: {e}")
                results[source_name] = pd.DataFrame()
        
        return results
    
    def apply_filters(self, data: pd.DataFrame, filters: Dict) -> pd.DataFrame:
        """Apply filters to a DataFrame"""
        if not filters or data.empty:
            return data
        
        filtered_data = data.copy()
        
        for key, value in filters.items():
            if key in filtered_data.columns:
                if isinstance(value, list):
                    filtered_data = filtered_data[filtered_data[key].isin(value)]
                else:
                    filtered_data = filtered_data[filtered_data[key] == value]
        
        return filtered_data
    
    def generate_response(self, query: str, parsed_query: Dict, 
                         semantic_results: List[Tuple[str, float]], 
                         structured_results: Dict[str, Any]) -> Dict[str, Any]:
        """Generate a comprehensive response to the query"""
        response = {
            "query": query,
            "query_type": parsed_query['type'],
            "timestamp": datetime.now().isoformat(),
            "results": {},
            "summary": "",
            "sources": []
        }
        
        # Process structured results
        all_wells = []
        for source_name, result in structured_results.items():
            # Handle both DataFrame and dict results
            if isinstance(result, pd.DataFrame) and not result.empty:
                response['results'][source_name] = result.to_dict('records')
                all_wells.extend(result.to_dict('records'))
            elif isinstance(result, dict) and result:
                response['results'][source_name] = result
                # Convert dict to list of records for consistency
                for well_id, well_data in result.items():
                    well_data['well_id'] = well_id
                    all_wells.append(well_data)
        
        # Process semantic results
        response['semantic_results'] = [
            {"document": doc[:200] + "..." if len(doc) > 200 else doc, "relevance_score": score}
            for doc, score in semantic_results
        ]
        
        # Generate summary based on query type
        response['summary'] = self.generate_summary(query, parsed_query, all_wells, semantic_results)
        
        # Add data sources
        response['sources'] = list(structured_results.keys())
        
        return response
    
    def generate_summary(self, query: str, parsed_query: Dict, 
                        wells: List[Dict], semantic_results: List[Tuple[str, float]]) -> str:
        """Generate a natural language summary of the results"""
        if not wells and not semantic_results:
            return "No relevant information found for your query."
        
        summary_parts = []
        
        # Handle different query types
        if parsed_query['type'] == 'location':
            if wells:
                summary_parts.append(f"Found {len(wells)} wells matching your location query:")
                for well in wells[:5]:  # Limit to top 5
                    well_name = well.get('name', well.get('well_name', 'Unknown'))
                    x_coord = well.get('x_coord', 'N/A')
                    y_coord = well.get('y_coord', 'N/A')
                    sector = well.get('sector', 'N/A')
                    summary_parts.append(f"- {well_name} at (X: {x_coord}, Y: {y_coord}) in Sector {sector}")
                if len(wells) > 5:
                    summary_parts.append(f"... and {len(wells) - 5} more wells.")
            else:
                summary_parts.append("No specific well locations found, but here's related information:")
                for doc, score in semantic_results[:3]:
                    summary_parts.append(f"- {doc[:100]}... (relevance: {score:.2f})")
        
        elif parsed_query['type'] == 'status':
            status_counts = {}
            for well in wells:
                status = well.get('status', 'unknown')
                status_counts[status] = status_counts.get(status, 0) + 1
            
            if status_counts:
                summary_parts.append("Well status summary:")
                for status, count in status_counts.items():
                    summary_parts.append(f"- {count} wells with status '{status}'")
            else:
                summary_parts.append("No status information found for the specified criteria.")
        
        elif parsed_query['type'] == 'production':
            producing_wells = [w for w in wells if w.get('production_rate', 0) > 0]
            if producing_wells:
                total_production = sum(w.get('production_rate', 0) for w in producing_wells)
                summary_parts.append(
                    f"Found {len(producing_wells)} producing wells with total production "
                    f"of {total_production} BOPD."
                )
            else:
                summary_parts.append("No producing wells found matching your criteria.")
        
        else:  # General query
            if wells:
                summary_parts.append(f"Found {len(wells)} wells matching your query.")
                # Add some key details from the first few wells
                for well in wells[:3]:
                    well_name = well.get('name', well.get('well_name', 'Unknown'))
                    well_type = well.get('well_type', 'N/A')
                    sector = well.get('sector', 'N/A')
                    summary_parts.append(f"- {well_name} ({well_type}) in Sector {sector}")
            else:
                summary_parts.append("Based on the available information:")
                for doc, score in semantic_results[:3]:
                    summary_parts.append(f"- {doc[:100]}... (relevance: {score:.2f})")
        
        return "\n".join(summary_parts)


class CSVDataConnector:
    """CSV data connector"""
    
    def __init__(self, data_path):
        self.data_path = data_path
    
    def load_data(self):
        try:
            return pd.read_csv(self.data_path)
        except FileNotFoundError:
            print(f"CSV file not found: {self.data_path}")
            return pd.DataFrame()
    
    def query(self, filters):
        data = self.load_data()
        if data.empty:
            return data
        
        filtered_data = data.copy()
        for key, value in filters.items():
            if key in filtered_data.columns:
                filtered_data = filtered_data[filtered_data[key] == value]
        
        return filtered_data
    
    def create_sample_data(self):
        """Create sample CSV data"""
        sample_data = {
            'well_id': ['A-1', 'B-3', 'G-5', 'D-2', 'E-7', 'Z-4'],
            'name': ['Alpha-1', 'Beta-3', 'Gamma-5', 'Delta-2', 'Epsilon-7', 'Zeta-4'],
            'x_coord': [1250, 1890, 2150, 1420, 2310, 1980],
            'y_coord': [3870, 4150, 3520, 3980, 4210, 3670],
            'sector': ['B-12', 'C-8', 'A-5', 'B-15', 'D-3', 'A-8'],
            'drill_date': ['2023-01-15', '2023-11-30', '2022-08-20', '2023-09-10', '2023-12-05', '2024-01-20'],
            'well_type': ['production', 'production', 'injection', 'monitoring', 'injection', 'production'],
            'status': ['active', 'active', 'active', 'active', 'active', 'drilling'],
            'production_rate': [1200, 1500, 800, 0, 600, 0],
            'reservoir': ['Sandstone', 'Limestone', 'Shale', 'Sandstone', 'Limestone', 'Sandstone']
        }
        
        df = pd.DataFrame(sample_data)
        df.to_csv(self.data_path, index=False)
        print(f"Created sample CSV data at {self.data_path}")


class SQLiteDataConnector:
    """SQLite data connector"""
    
    def __init__(self, data_path):
        self.data_path = data_path
        self.connection = None
    
    def connect(self):
        """Establish database connection"""
        if self.connection is None:
            self.connection = sqlite3.connect(self.data_path)
        return self.connection
    
    def load_data(self):
        try:
            conn = self.connect()
            return pd.read_sql_query("SELECT * FROM wells", conn)
        except sqlite3.OperationalError:
            print(f"SQLite table not found: {self.data_path}")
            return pd.DataFrame()
    
    def query(self, filters):
        data = self.load_data()
        if data.empty:
            return data
        
        filtered_data = data.copy()
        for key, value in filters.items():
            if key in filtered_data.columns:
                filtered_data = filtered_data[filtered_data[key] == value]
        
        return filtered_data
    
    def create_sample_data(self):
        """Create sample SQLite data"""
        conn = self.connect()
        cursor = conn.cursor()
        
        # Create table
        cursor.execute('''
        CREATE TABLE IF NOT EXISTS wells (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            well_id TEXT,
            name TEXT,
            x_coord REAL,
            y_coord REAL,
            sector TEXT,
            drill_date TEXT,
            well_type TEXT,
            status TEXT,
            production_rate REAL,
            reservoir TEXT
        )
        ''')
        
        # Insert sample data
        sample_data = [
            ('A-1', 'Alpha-1', 1250, 3870, 'B-12', '2023-01-15', 'production', 'active', 1200, 'Sandstone'),
            ('B-3', 'Beta-3', 1890, 4150, 'C-8', '2023-11-30', 'production', 'active', 1500, 'Limestone'),
            ('G-5', 'Gamma-5', 2150, 3520, 'A-5', '2022-08-20', 'injection', 'active', 800, 'Shale'),
            ('D-2', 'Delta-2', 1420, 3980, 'B-15', '2023-09-10', 'monitoring', 'active', 0, 'Sandstone'),
            ('E-7', 'Epsilon-7', 2310, 4210, 'D-3', '2023-12-05', 'injection', 'active', 600, 'Limestone'),
            ('Z-4', 'Zeta-4', 1980, 3670, 'A-8', '2024-01-20', 'production', 'drilling', 0, 'Sandstone')
        ]
        
        cursor.executemany('''
        INSERT OR IGNORE INTO wells (well_id, name, x_coord, y_coord, sector, drill_date, well_type, status, production_rate, reservoir)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', sample_data)
        
        conn.commit()
        print(f"Created sample SQLite data at {self.data_path}")


class JSONDataConnector:
    """JSON data connector for additional metadata"""
    
    def __init__(self, data_path):
        self.data_path = data_path
    
    def load_data(self):
        try:
            with open(self.data_path, 'r') as f:
                return json.load(f)
        except FileNotFoundError:
            print(f"JSON file not found: {self.data_path}")
            return {}
    
    def query(self, filters):
        data = self.load_data()
        if not data:
            return {}
        
        # Simple filtering for JSON data
        filtered_data = {}
        for well_id, metadata in data.items():
            # Check if any filter matches
            match = True
            for key, value in filters.items():
                if key in metadata and metadata[key] != value:
                    match = False
                    break
            
            if match:
                filtered_data[well_id] = metadata
        
        return filtered_data
    
    def create_sample_data(self):
        """Create sample JSON metadata"""
        sample_data = {
            "A-1": {
                "description": "Primary production well in the northern sector",
                "completion_data": "Completed with 5-1/2 inch casing",
                "production_history": "Stable production since completion",
                "geological_data": "Targets the Upper Sandstone formation"
            },
            "B-3": {
                "description": "High-rate production well with ESP",
                "completion_data": "Gravel packed completion",
                "production_history": "Increasing production trend",
                "geological_data": "Limestone reservoir with good porosity"
            },
            "G-5": {
                "description": "Water injection well for pressure maintenance",
                "completion_data": "Dual completion string",
                "production_history": "Injection started Q2 2022",
                "geological_data": "Shale formation with natural fractures"
            }
        }
        
        with open(self.data_path, 'w') as f:
            json.dump(sample_data, f, indent=2)
        
        print(f"Created sample JSON data at {self.data_path}")


def main():
    """Main function to demonstrate the RAG system"""
    print("=" * 80)
    print("BROWNFIELD RAG SYSTEM FOR WELL MANAGEMENT")
    print("=" * 80)
    
    # Initialize the RAG system
    rag_system = BrownfieldRAGSystem()
    
    # Example queries
    queries = [
        "Where are the production wells located?",
        "Show me active wells in Sector B",
        "What's the status of well Gamma-5?",
        "Find injection wells drilled in 2023",
        "Where are the new wells located?",
        "Show me wells in Sandstone reservoir",
        "What's the production rate of Beta-3?",
        "Find monitoring wells with production over 1000 barrels",
        "Which wells were completed recently?",
        "Show me all wells with their coordinates"
    ]
    
    # Process each query
    for i, query in enumerate(queries, 1):
        print(f"\n{'='*80}")
        print(f"QUERY {i}: {query}")
        print(f"{'='*80}")
        
        try:
            response = rag_system.process_query(query)
            
            # Print the summary
            print(f"SUMMARY:\n{response['summary']}\n")
            
            # Print additional details if available
            if response['results']:
                print("DETAILED RESULTS:")
                for source, results in response['results'].items():
                    if results:
                        print(f"\nFrom {source}:")
                        if isinstance(results, list):
                            for j, result in enumerate(results[:3], 1):  # Show first 3 results
                                print(f"  {j}. {result.get('name', result.get('well_name', 'Unknown'))}")
                                if 'x_coord' in result and 'y_coord' in result:
                                    print(f"     Location: (X: {result['x_coord']}, Y: {result['y_coord']})")
                                if 'production_rate' in result:
                                    print(f"     Production: {result['production_rate']} BOPD")
                        elif isinstance(results, dict):
                            for j, (well_id, metadata) in enumerate(list(results.items())[:3], 1):
                                print(f"  {j}. Well {well_id}:")
                                for key, value in metadata.items():
                                    print(f"     {key}: {value}")
            
            # Print semantic results if no structured results
            if not response['results'] and response.get('semantic_results'):
                print("\nSEMANTIC SEARCH RESULTS:")
                for j, result in enumerate(response['semantic_results'][:3], 1):
                    print(f"  {j}. [Relevance: {result['relevance_score']:.3f}] {result['document']}")
                    
        except Exception as e:
            print(f"Error processing query: {e}")
            import traceback
            traceback.print_exc()


if __name__ == "__main__":
    # Install required packages if not already installed
    try:
        import pandas as pd
        import spacy
    except ImportError:
        print("Please install required packages:")
        print("pip install pandas numpy scikit-learn spacy")
        exit(1)
    
    # Run the main demonstration
    main()

BROWNFIELD RAG SYSTEM FOR WELL MANAGEMENT

------------------------------------------------------------
Initializing Brownfield RAG System
------------------------------------------------------------
Created data directories in data
✓ Loaded spaCy model: en_core_web_sm

------------------------------------------------------------
Initializing data sources
------------------------------------------------------------
Created sample CSV data at data/csv/well_data.csv
Created sample SQLite data at data/sqlite/brownfield_wells.db
Created sample JSON data at data/json/well_metadata.json

------------------------------------------------------------
Preparing document corpus
------------------------------------------------------------
Loaded 6 records from CSV, 18 from SQLite, 3 from JSON
Created 27 document representations
Document vectorization completed

------------------------------------------------------------
Brownfield RAG System initialized successfully
------------------------------